In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

ModuleNotFoundError: No module named 'torch'

In [2]:
def make_windows(series, L, H):
    X =[]
    Y =[]
    for i in range(len(series) - L - H + 1):
        X.append(series[i: i+ L])
        Y.append(series[i+L: i+L+H])
    return  np.array(X), np.array(Y)

x,y = make_windows(np.arange(10), 3, 2)
print(x.shape, y.shape)

(6, 3) (6, 2)


In [3]:
def ols(x,y):
    A = x.T @ x
    C = x.T @ y
    return np.linalg.solve(A,C).T

rng = np.random.default_rng(0)
W_true = rng.normal(size=(2, 3))
X_toy = rng.normal(size=(100, 3))
Y_toy = X_toy @ W_true.T          # dữ liệu sinh đúng từ W_true, không nhiễu

W_hat = ols(X_toy, Y_toy)
print(np.abs(W_hat - W_true).max())

2.220446049250313e-16


In [4]:
def ridge(X, Y, lam):
    A =( X.T @ X) + (lam * np.eye(X.shape[1]))
    C = X.T @ Y
    return np.linalg.solve(A,C).T

np.abs(ridge(X_toy, Y_toy, 0) - ols(X_toy, Y_toy)).max()

np.float64(0.0)

In [5]:
def nlinear_constrained(X, Y):
    w_ols = ols(X,Y)
    L = np.ones(X.shape[1])
    H = np.ones(Y.shape[1])
    u = np.linalg.solve(X.T @ X, L)
    s = L.T @ u
    V= w_ols - ((1/s) * np.outer((w_ols @ L - H) , u))
    return V


nlinear_constrained(X_toy, Y_toy)

array([[ 0.22505322, -0.00703769,  0.78198446],
       [ 0.3950846 , -0.17027008,  0.77518548]])

In [6]:
def mse(W, X, Y):
    return np.mean((Y - X @ W.T) ** 2)

# W_ols = ols(X_toy, Y_toy)
# V     = nlinear_constrained(X_toy, Y_toy)
# print(mse(W_ols, X_toy, Y_toy))
# print(mse(V,     X_toy, Y_toy))

In [7]:
def fit_with_bias(solver, X, Y):
    x_bar = X.mean(axis=0)
    y_bar = Y.mean(axis=0)
    X_c   = X - x_bar
    Y_c   = Y - y_bar
    W     = solver(X_c, Y_c)
    b     = y_bar - W @ x_bar
    return W, b
# W, b = fit_with_bias(ols, X_toy, Y_b)   # truyền ols, không có ngoặc
# print(np.abs(W - W_true).max(), np.abs(b - b_true).max())


In [8]:
def load_etth1(path, L):
    df   = pd.read_csv(path)
    data = df.iloc[:, 1:].values
    mean = np.mean(data[:8640], axis=0)
    std  = np.std(data[:8640], axis=0)
    data = (data - mean) / std
    train = data[0 : 8640]
    val   = data[8640 - L : 11520]
    test  = data[11520 - L : 14400]
    return train, val, test

In [9]:
def make_dataset(split, L, H):
    Xs, Ys = [], []
    for c in range(split.shape[1]):
        X_c, Y_c = make_windows(split[:, c], L, H)
        Xs.append(X_c)
        Ys.append(Y_c)
    return np.concatenate(Xs, axis=0), np.concatenate(Ys, axis=0)

In [10]:
L, H = 336, 96
train, val, test = load_etth1("../data/ETTh1.csv", L)
X_train, Y_train = make_dataset(train, L, H)
X_test,  Y_test  = make_dataset(test,  L, H)
print(X_train.shape, X_test.shape)

W, b = fit_with_bias(ols, X_train, Y_train)
pred =   X_test @ W.T + b                 
print(np.mean((Y_test - pred) ** 2))


(57463, 336) (19495, 336)
0.37023527067813916


In [11]:
# --- 0. Hàm loss có bias (hàm mse cũ thiếu b) ---
def sse(W, b, X, Y):
    # TODO: tổng (không phải trung bình) bình phương phần dư, dự báo = X @ W.T + b
    pred = X @ (W.T) + b
    return np.sum((Y - pred) ** 2)


W_ols, b_ols = fit_with_bias(ols, X_train, Y_train)
V,     b_V   = fit_with_bias(nlinear_constrained, X_train, Y_train)
lhs = sse(V, b_V, X_train, Y_train) - sse(W_ols, b_ols, X_train, Y_train)  # TODO: hiệu SSE train, NLinear trừ OLS

# --- 2. Vế phải: chỉ từ định nghĩa, KHÔNG dùng V hay b_V ---
X_c = X_train - X_train.mean(axis = 0)   # TODO: X_train trừ trung bình cột
A   =( X_c.T @ X_c) # TODO
L = np.ones(336)
u = np.linalg.solve(X_c.T @ X_c, L)
s   = np.ones(X_train.shape[1]).T @ u   # TODO: s = 1_L^T u
r   = W_ols@np.ones(X_c.shape[1]) - np.ones(Y_train.shape[1])    # TODO: r = W_ols 1_L − 1_H
rhs =  np.linalg.norm(r)**2 /s   # TODO: ||r||^2 / s


print(lhs, rhs, abs(lhs - rhs) / rhs)

print(lhs) 
print(u.shape)

17002.829247226007 17002.82924722584 9.842304636066549e-15
17002.829247226007
(336,)


In [12]:
lhs/Y_train.size

np.float64(0.0030822060222857185)

In [13]:
print(sse(V, b_V,X_test, Y_test)/Y_test.size)

0.3701986328402697


In [14]:
mae_ols = np.mean(np.abs(W_ols.sum(axis=1) - 1).max())
mae_V   = np.mean(np.abs(pred_V.sum(axis= 1) -1).max())

NameError: name 'pred_V' is not defined